# crop_for_manual_alignment
Saves each **AbPAS / DAPI pair** as two separate TIFF files at the same pixel
size and (approximate) field of view, ready for manual alignment in Fiji or ZEN.

**Pipeline at a glance**
1. Find AbPAS / DAPI `.czi` pairs
2. Read both files and inspect metadata
3. Resample DAPI to AbPAS pixel size
4. Crop DAPI to the AbPAS field of view using stage coordinates
5. Preview the aligned pair
6. Save calibrated TIFFs


In [3]:
import os, re, sys, glob, warnings

import numpy as np
import tifffile
from biochemical_stain_assessment.czi_io import read_czi
from scipy.ndimage import zoom as ndzoom

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

warnings.filterwarnings("ignore")

print("Imports OK")


Imports OK


## Configuration

In [4]:
# ── Edit these two paths ─────────────────────────────────────────────────────
DATA_DIR = os.path.dirname(os.path.abspath("crop_for_manual_alignment.ipynb"))
OUT_DIR  = os.path.join(DATA_DIR, "registered_output", "separate")

print(f"Scanning : {DATA_DIR}")
print(f"Output   : {OUT_DIR}")


Scanning : /Users/serenasritharan/Projects/biochemical-stain-assessment
Output   : /Users/serenasritharan/Projects/biochemical-stain-assessment/registered_output/separate


## Helper functions

In [ ]:
def resample(img, src_px, tgt_px):
    factor = src_px / tgt_px
    if abs(factor - 1.0) < 1e-4:
        return img
    src_dtype = img.dtype
    out = ndzoom(img.astype(np.float32), factor, order=1)
    info = np.iinfo(src_dtype) if np.issubdtype(src_dtype, np.integer)            else np.finfo(src_dtype)
    return np.clip(out, info.min, info.max).astype(src_dtype)


def crop_dapi_to_fov(dapi_rs, h_a, w_a, cx_a, cy_a, cx_d, cy_d, px):
    h_dr, w_dr = dapi_rs.shape[:2]
    dx_px =  (cx_a - cx_d) / px
    dy_px = -(cy_a - cy_d) / px
    r0 = int(round(h_dr / 2 + dy_px - h_a / 2))
    c0 = int(round(w_dr / 2 + dx_px - w_a / 2))
    r0 = max(0, r0);  c0 = max(0, c0)
    r1 = min(h_dr, r0 + h_a)
    c1 = min(w_dr, c0 + w_a)
    crop = dapi_rs[r0:r1, c0:c1]
    pr = h_a - crop.shape[0];  pc = w_a - crop.shape[1]
    if pr > 0 or pc > 0:
        crop = np.pad(crop, [(pr // 2, pr - pr // 2),
                             (pc // 2, pc - pc // 2)])
    return crop, r0, c0


def save_tiff(path, arr, px_um, channel_name):
    resolution = (1e4 / px_um, 1e4 / px_um)
    if arr.ndim == 2:
        tifffile.imwrite(path, arr, resolution=resolution,
                         resolutionunit=tifffile.RESUNIT.CENTIMETER,
                         compression='deflate', compressionargs={'level': 6},
                         metadata={'axes': 'YX'})
    else:
        tifffile.imwrite(path, arr, photometric='rgb', resolution=resolution,
                         resolutionunit=tifffile.RESUNIT.CENTIMETER,
                         compression='deflate', compressionargs={'level': 6},
                         metadata={'axes': 'YXS'})
    print(f"  → {os.path.basename(path)}  ({arr.shape}, {arr.dtype}, {px_um:.4f} µm/px)")


def find_pairs(data_dir):
    abpas_files = glob.glob(
        os.path.join(data_dir, '**', 'AbPAS', '*.czi'), recursive=True)
    pairs = []
    for ap in sorted(abpas_files):
        m = re.search(r'_AbPAS_(.+?)\.czi$', os.path.basename(ap))
        if not m:
            continue
        sample_id = m.group(1)
        parent = os.path.dirname(os.path.dirname(ap))
        matches = glob.glob(os.path.join(parent, 'DAPI', f'*_{sample_id}.czi'))
        if matches:
            pairs.append((sample_id, ap, matches[0]))
        else:
            print(f"  ⚠  No DAPI match for '{sample_id}' — skipping.")
    return pairs

print("Helper functions defined ✓")


## Step 1 — Discover AbPAS / DAPI pairs

In [ ]:
pairs = find_pairs(DATA_DIR)
print(f"Found {len(pairs)} pair(s):\n")
for i, (sid, ap, dp) in enumerate(pairs):
    print(f"  [{i}] {sid}")
    print(f"       AbPAS : {os.path.basename(ap)}")
    print(f"       DAPI  : {os.path.basename(dp)}")
    print()


## Step 2 — Read one pair and inspect metadata
Change `SAMPLE_IDX` to inspect a different pair.


In [ ]:
SAMPLE_IDX = 0

sample_id, abpas_path, dapi_path = pairs[SAMPLE_IDX]
print(f"Sample : {sample_id}")
print(f"AbPAS  : {abpas_path}")
print(f"DAPI   : {dapi_path}\n")

abpas, px_a, _, cx_a, cy_a, _ = read_czi(abpas_path)
dapi,  px_d, _, cx_d, cy_d, _ = read_czi(dapi_path)

print("─── AbPAS ───────────────────────────────────────────")
print(f"  Shape  : {abpas.shape}  dtype={abpas.dtype}")
print(f"  Pixel  : {px_a:.4f} µm/px")
print(f"  Centre : ({cx_a:.1f}, {cy_a:.1f}) µm")

print("\n─── DAPI ────────────────────────────────────────────")
print(f"  Shape  : {dapi.shape}  dtype={dapi.dtype}")
print(f"  Pixel  : {px_d:.4f} µm/px")
print(f"  Centre : ({cx_d:.1f}, {cy_d:.1f}) µm")

print(f"\n  DAPI/AbPAS pixel ratio : {px_d/px_a:.4f}  "
      f"(DAPI pixel is {px_d/px_a:.1f}× larger)")


## Step 2b — Visualise raw images (thumbnails at 1/32 resolution)

In [ ]:
DS = 32   # downsample factor for quick display

def thumb(img, ds):
    """Return a small version of img for display."""
    if img.ndim == 3:   # RGB
        return img[::ds, ::ds]
    else:               # grayscale
        return img[::ds, ::ds]

abpas_t = thumb(abpas, DS)
dapi_t  = thumb(dapi,  DS)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(abpas_t)
axes[0].set_title(f"AbPAS (raw)\n{abpas.shape}  {px_a:.4f} µm/px", fontsize=11)
axes[0].axis('off')

axes[1].imshow(dapi_t, cmap='magma')
axes[1].set_title(f"DAPI (raw)\n{dapi.shape}  {px_d:.4f} µm/px", fontsize=11)
axes[1].axis('off')

fig.suptitle(f"Sample: {sample_id} — raw images", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"AbPAS physical size : {abpas.shape[1]*px_a/1000:.2f} mm × "
      f"{abpas.shape[0]*px_a/1000:.2f} mm")
print(f"DAPI  physical size : {dapi.shape[1]*px_d/1000:.2f} mm × "
      f"{dapi.shape[0]*px_d/1000:.2f} mm")


## Step 3 — Resample DAPI to AbPAS pixel size

In [ ]:
print(f"Resampling DAPI: {px_d:.4f} → {px_a:.4f} µm/px  "
      f"(factor x{px_d/px_a:.3f}) …")

dapi_rs = resample(dapi, px_d, px_a)

print(f"  DAPI before : {dapi.shape}")
print(f"  DAPI after  : {dapi_rs.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(dapi_t, cmap='magma')
axes[0].set_title(f"DAPI original\n{dapi.shape}  {px_d:.4f} µm/px", fontsize=11)
axes[0].axis('off')

axes[1].imshow(dapi_rs[::DS, ::DS], cmap='magma')
axes[1].set_title(f"DAPI resampled\n{dapi_rs.shape}  {px_a:.4f} µm/px", fontsize=11)
axes[1].axis('off')

fig.suptitle("Resampling: DAPI pixel size matched to AbPAS",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Show the size difference relative to AbPAS canvas
h_a, w_a = abpas.shape[:2]
h_dr, w_dr = dapi_rs.shape[:2]
print(f"\nAbPAS canvas          : {w_a} × {h_a} px")
print(f"Resampled DAPI canvas : {w_dr} × {h_dr} px")
print(f"DAPI is larger by     : {w_dr-w_a} cols, {h_dr-h_a} rows")


## Step 4 — Crop DAPI to the AbPAS field of view using stage coordinates

In [ ]:
dapi_crop, r0, c0 = crop_dapi_to_fov(
    dapi_rs, h_a, w_a, cx_a, cy_a, cx_d, cy_d, px_a)

print(f"Stage offset AbPAS vs DAPI:")
print(f"  ΔX = {cx_a - cx_d:+.1f} µm  →  {(cx_a-cx_d)/px_a:+.1f} px (cols)")
print(f"  ΔY = {cy_a - cy_d:+.1f} µm  →  {-(cy_a-cy_d)/px_a:+.1f} px (rows, Y inverted)")
print(f"\nCrop origin in resampled DAPI : row={r0}, col={c0}")
print(f"Cropped DAPI shape            : {dapi_crop.shape}")
print(f"Target AbPAS shape            : ({h_a}, {w_a})")
match = dapi_crop.shape[:2] == (h_a, w_a)
print(f"Shape match                   : {'✓' if match else '✗'}")


### Visualise the crop window inside the resampled DAPI

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Left: resampled DAPI with crop rectangle
dapi_rs_t = dapi_rs[::DS, ::DS]
axes[0].imshow(dapi_rs_t, cmap='magma')
rect = mpatches.Rectangle(
    (c0 / DS, r0 / DS), w_a / DS, h_a / DS,
    linewidth=2, edgecolor='cyan', facecolor='none',
    label='AbPAS FOV')
axes[0].add_patch(rect)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].set_title(f"Resampled DAPI\n(cyan box = AbPAS FOV)", fontsize=11)
axes[0].axis('off')

# Centre: cropped DAPI
axes[1].imshow(dapi_crop[::DS, ::DS], cmap='magma')
axes[1].set_title(f"DAPI cropped to AbPAS FOV\n{dapi_crop.shape}", fontsize=11)
axes[1].axis('off')

# Right: AbPAS
axes[2].imshow(abpas_t)
axes[2].set_title(f"AbPAS\n{abpas.shape}", fontsize=11)
axes[2].axis('off')

fig.suptitle(f"Sample: {sample_id} — field-of-view alignment",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 5 — Quick overlay preview

In [ ]:
def make_overlay(abpas_rgb, dapi_arr, ds=DS, alpha=0.4):
    """Blend DAPI (cyan) onto AbPAS colour at thumbnail resolution."""
    ab = abpas_rgb[::ds, ::ds].astype(np.float32) / 255.0
    d  = dapi_arr[::ds, ::ds].astype(np.float32)
    d  = (d - d.min()) / (d.max() - d.min() + 1e-9)

    comp = ab.copy()
    comp[:, :, 0] = np.clip(comp[:, :, 0] - alpha * d, 0, 1)  # subtract red
    comp[:, :, 1] = np.clip(comp[:, :, 1] + alpha * d, 0, 1)
    comp[:, :, 2] = np.clip(comp[:, :, 2] + alpha * d, 0, 1)
    return comp

overlay = make_overlay(abpas, dapi_crop)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(abpas_t)
axes[0].set_title("AbPAS (brightfield)", fontsize=11)
axes[0].axis('off')

axes[1].imshow(dapi_crop[::DS, ::DS], cmap='magma')
axes[1].set_title("DAPI (fluorescence)", fontsize=11)
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title("Overlay  (DAPI in cyan, α=0.4)", fontsize=11)
axes[2].axis('off')

# Legend
cyan_patch  = mpatches.Patch(color='cyan',  label='DAPI / Hoechst')
green_patch = mpatches.Patch(color='gray',  label='AbPAS brightfield')
axes[2].legend(handles=[cyan_patch, green_patch],
               loc='lower right', fontsize=9)

fig.suptitle(f"Sample: {sample_id} — aligned overlay",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Step 6 — Save all pairs as calibrated TIFFs

In [ ]:
def process_pair(sample_id, abpas_path, dapi_path, out_dir):
    print(f"\n  {sample_id}")
    abpas, px_a, _, cx_a, cy_a, _ = read_czi(abpas_path)
    dapi,  px_d, _, cx_d, cy_d, _ = read_czi(dapi_path)
    h_a, w_a = abpas.shape[:2]

    dapi_rs   = resample(dapi, px_d, px_a)
    dapi_crop, _, _ = crop_dapi_to_fov(dapi_rs, h_a, w_a,
                                        cx_a, cy_a, cx_d, cy_d, px_a)
    os.makedirs(out_dir, exist_ok=True)
    save_tiff(os.path.join(out_dir, f"{sample_id}_AbPAS.tif"), abpas, px_a, "AbPAS")
    save_tiff(os.path.join(out_dir, f"{sample_id}_DAPI.tif"),  dapi_crop, px_a, "DAPI")

print(f"Processing {len(pairs)} pair(s) → {OUT_DIR}\n")
for sid, ap, dp in pairs:
    process_pair(sid, ap, dp, OUT_DIR)

print(f"\nDone — TIFFs saved to: {OUT_DIR}")
print("\nTo align in Fiji:")
print("  1. File → Open both *_AbPAS.tif and *_DAPI.tif")
print("  2. Image → Color → Merge Channels (AbPAS → gray, DAPI → blue/cyan)")
print("  3. Use StackReg or BigWarp for fine-tuning")
